<a href="https://colab.research.google.com/github/Mohanapriya2210/Finalyearproject/blob/main/UI_Frame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit -q
!npm install -g localtunnel -q

from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_DIR = "/content/drive/MyDrive/DR_MODEL"  # Update this if needed
os.environ['MODEL_DIR'] = MODEL_DIR

  Stopping...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 3s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install streamlit numpy opencv-python scikit-learn joblib tensorflow xgboost -q
print("✅ All packages installed!")

✅ All packages installed!


In [ ]:
from tensorflow.keras import config
config.enable_unsafe_deserialization()

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

# ⚠️ IMPORTANT: Enable unsafe deserialization for Lambda layers
from tensorflow.keras import config
config.enable_unsafe_deserialization()

st.set_page_config(page_title="DR Predictor", layout="centered")
st.title("🩺 Diabetic Retinopathy Predictor")

MODEL_DIR = os.environ.get("MODEL_DIR", "/content/drive/MyDrive/DR_MODEL")

# Load models
@st.cache_resource
def load_all_models():
    try:
        st.info("Loading models...")

        # Check if files exist
        required_files = [
            "tab_embedding_model.keras",
            "fusion_model.keras",
            "xgb_model.pkl",
            "tab_scaler.pkl",
            "fusion_scaler.pkl"
        ]

        for file in required_files:
            if not os.path.exists(os.path.join(MODEL_DIR, file)):
                return False, f"Missing file: {file}"

        # Load all models - use safe_mode=False for Lambda layers
        tab_model = load_model(
            os.path.join(MODEL_DIR, "tab_embedding_model.keras"),
            safe_mode=False  # Added this
        )
        fusion_model = load_model(
            os.path.join(MODEL_DIR, "fusion_model.keras"),
            safe_mode=False  # Added this
        )
        xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb_model.pkl"))
        tab_scaler = joblib.load(os.path.join(MODEL_DIR, "tab_scaler.pkl"))
        fusion_scaler = joblib.load(os.path.join(MODEL_DIR, "fusion_scaler.pkl"))

        # Load EfficientNet
        base_model = EfficientNetV2S(
            include_top=False,
            weights="imagenet",
            input_shape=(224, 224, 3)
        )
        for layer in base_model.layers:
            layer.trainable = False

        x = GlobalAveragePooling2D()(base_model.output)
        feature_extractor = Model(base_model.input, x)

        return True, {
            "tab_model": tab_model,
            "fusion_model": fusion_model,
            "xgb_model": xgb_model,
            "tab_scaler": tab_scaler,
            "fusion_scaler": fusion_scaler,
            "feature_extractor": feature_extractor
        }

    except Exception as e:
        import traceback
        return False, f"{str(e)}\n\n{traceback.format_exc()}"

# Load models
success, result = load_all_models()

if not success:
    st.error(f"❌ Model loading failed: {result}")

    # Try alternative loading method
    st.info("⚠️ Trying alternative loading method...")
    try:
        import tensorflow as tf

        # Try loading with custom_objects
        tab_model = tf.keras.models.load_model(
            os.path.join(MODEL_DIR, "tab_embedding_model.keras"),
            compile=False
        )
        fusion_model = tf.keras.models.load_model(
            os.path.join(MODEL_DIR, "fusion_model.keras"),
            compile=False
        )

        st.success("✅ Models loaded with alternative method!")

        # Load other components
        xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb_model.pkl"))
        tab_scaler = joblib.load(os.path.join(MODEL_DIR, "tab_scaler.pkl"))
        fusion_scaler = joblib.load(os.path.join(MODEL_DIR, "fusion_scaler.pkl"))

        # Load EfficientNet
        base_model = EfficientNetV2S(
            include_top=False,
            weights="imagenet",
            input_shape=(224, 224, 3)
        )
        for layer in base_model.layers:
            layer.trainable = False

        x = GlobalAveragePooling2D()(base_model.output)
        feature_extractor = Model(base_model.input, x)

        success = True
        models = {
            "tab_model": tab_model,
            "fusion_model": fusion_model,
            "xgb_model": xgb_model,
            "tab_scaler": tab_scaler,
            "fusion_scaler": fusion_scaler,
            "feature_extractor": feature_extractor
        }

    except Exception as e2:
        st.error(f"❌ Alternative loading also failed: {e2}")
        st.stop()
else:
    st.success("✅ Models loaded successfully!")
    models = result

# UI Components
uploaded_file = st.file_uploader("Upload Retinal Image", type=["jpg", "png", "jpeg"])

col1, col2 = st.columns(2)
with col1:
    glucose = st.number_input("Glucose (mg/dL)", 50, 300, 120)
    bp = st.number_input("Blood Pressure (mmHg)", 30, 150, 70)
    bmi = st.number_input("BMI", 10.0, 60.0, 25.0, step=0.1)
    age = st.number_input("Age", 10, 90, 35)

with col2:
    preg = st.number_input("Pregnancies", 0, 20, 1)
    insulin = st.number_input("Insulin (µU/mL)", 10, 900, 80)
    dpf = st.number_input("Diabetes Pedigree Function", 0.05, 2.0, 0.5, step=0.01)
    skin = st.number_input("Skin Thickness (mm)", 5, 100, 20)

# Prediction button
if st.button("🔍 Predict DR Severity", type="primary"):
    if uploaded_file is None:
        st.warning("Please upload a retinal image.")
    else:
        with st.spinner("Processing..."):
            try:
                # Read and process image
                file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
                img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Display original
                st.image(img, caption="Uploaded Image", width=300)

                # Preprocess for model
                img_resized = cv2.resize(img, (224, 224))
                img_normalized = img_resized / 255.0
                img_batch = np.expand_dims(img_normalized, axis=0)

                # Extract image features
                img_feat = models["feature_extractor"].predict(img_batch, verbose=0)

                # Process tabular data
                tab_input = np.array([[preg, glucose, bp, skin, insulin, bmi, dpf, age]])
                tab_scaled = models["tab_scaler"].transform(tab_input)
                tab_embed = models["tab_model"].predict(tab_scaled, verbose=0)

                # Fusion
                fused = models["fusion_model"].predict([img_feat, tab_embed], verbose=0)
                fused_scaled = models["fusion_scaler"].transform(fused)

                # Prediction
                pred = int(models["xgb_model"].predict(fused_scaled)[0])

                # Display result
                severity_map = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]
                colors = ["green", "blue", "orange", "red", "darkred"]

                st.markdown(f"""
                <div style='padding: 20px; background-color: {colors[pred]}20; border-radius: 10px; border-left: 5px solid {colors[pred]};'>
                    <h2 style='color: {colors[pred]};'>Prediction: Stage {pred}</h2>
                    <h3>{severity_map[pred]}</h3>
                </div>
                """, unsafe_allow_html=True)

                # Show processed image
                st.image(img_resized, caption="Processed Image (224x224)", width=250)

            except Exception as e:
                st.error(f"Error during prediction: {str(e)}")

# Footer
st.markdown("---")
st.info("⚠️ This tool is for educational purposes only. Always consult a healthcare professional.")

Overwriting app.py


In [ ]:
from google.colab.output import eval_js
from IPython.display import display, HTML
import subprocess
import threading
import time
import urllib.parse

# Start Streamlit
def run_streamlit():
    !streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false 2>&1

# Start in background
thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for server to start
time.sleep(5)

# Get Colab proxy URL
print("⏳ Setting up Colab proxy...")
public_url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"✅ Done! Your app is ready at:")
print(f"\n🔗 {public_url}")
print("\n📱 Click the link above or copy it to your browser!")




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.242.232:8501

⏳ Setting up Colab proxy...


KeyboardInterrupt: 